# 08b · Fine-tune DistilBERT on Colab: BankSight

**What "fine-tuning" means:** DistilBERT (and RoBERTa) are neural networks that already understands English (it was pre-trained on a huge amount of text).
We show it our complaints with their outcomes (relief or no relief) and adjust it slightly so it predicts relief. It is a *transfer learning* recipe.

**What this notebook does**
1. Loads the two files you uploaded to Google Drive (train and validation text).
2. Trains the chosen model (DistilBERT first, then optionally the stronger RoBERTa) for 2 epochs (an *epoch* is one full pass over the training data), with a plain, readable training loop.
3. After each epoch, predicts the validation complaints and saves the probabilities to Drive.


**What this does:** checks that a GPU is attached. If it says False, change the runtime type (Runtime menu) and restart.

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

**What this does:** installs the `transformers` library if Colab does not already have it. (Run it once; ignore warnings.)

In [ ]:
# %pip install -q transformers

**What this does:** loads the libraries.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

**What this does:** connects Google Drive so our files and results survive if Colab disconnects. (A pop-up will ask for permission.)
If this code is run somewhere that is not Colab, it falls back to the local `data/colab` folder.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/banksight_colab")
except ImportError:
    DATA_DIR = Path("../data/colab")

OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data folder:", DATA_DIR)
print("files there:", [p.name for p in DATA_DIR.glob("*.parquet")])

**What this does:** the settings, all in one place. Pick the model with `WHICH_MODEL`:
* `"distilbert"`: small and fast (about 20 minutes on a free T4 GPU). Run this **first**: it proves the whole pipeline works.
* `"roberta"`: RoBERTa-base, about twice the size and usually stronger on text (about 60 to 90 minutes on a T4). Run it **second**, after DistilBERT works.

`MAX_LENGTH` is how many word-pieces of each complaint the model reads (longer text is cut off). `BATCH_SIZE` is how many complaints are processed
at once (RoBERTa is bigger, so it uses a smaller batch to fit in GPU memory).

In [ ]:
WHICH_MODEL = "distilbert"   # change to "roberta" for the stronger model, then restart the runtime and run everything again

if WHICH_MODEL == "distilbert":
    MODEL_NAME = "distilbert-base-uncased"
    BATCH_SIZE = 32
elif WHICH_MODEL == "roberta":
    MODEL_NAME = "roberta-base"
    BATCH_SIZE = 16

MODEL_TAG = WHICH_MODEL   # used in the saved file names, so the two models never overwrite each other
MAX_LENGTH = 256
EPOCHS = 2
LEARNING_RATE = 2e-5
WARMUP_SHARE = 0.06      # the learning rate rises slowly for the first 6% of steps, then falls
SEED = 42
MAX_TRAIN_ROWS = None    # set to e.g. 20000 for a quick trial run; None means use all training rows

np.random.seed(SEED)
torch.manual_seed(SEED)

**What this does:** loads the training and validation text.

In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
valid = pd.read_parquet(DATA_DIR / "validation.parquet")
if MAX_TRAIN_ROWS is not None:
    train = train.tail(MAX_TRAIN_ROWS).reset_index(drop=True)

print(f"train {len(train):,} | validation {len(valid):,}")
print(f"relief rate: train {train['label'].mean():.3f} | validation {valid['label'].mean():.3f}")

**What this does:** loads DistilBERT and its *tokenizer* (the tool that cuts text into word-pieces and turns them into numbers). The model gets a
single output number per complaint: a score that we turn into a probability with the sigmoid function.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
model.to(device)
print("model loaded on", device)

**What this does:** sets up the optimiser (AdamW, the standard choice for transformers), the learning-rate schedule, the loss function
(binary cross-entropy: how wrong the predicted probability is) and mixed precision (half-size numbers on the GPU, for speed).

In [ ]:
steps_per_epoch = int(np.ceil(len(train) / BATCH_SIZE))
total_steps = steps_per_epoch * EPOCHS

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_SHARE * total_steps), total_steps)
loss_function = torch.nn.BCEWithLogitsLoss()

use_gpu = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_gpu)

epoch_number = 0
print(f"{steps_per_epoch} steps per epoch, {total_steps} steps in total")

## Train: run the next two cells together, then run them again for epoch 2
Training and prediction are separate cells so you can see each epoch's result, and so you can resume if Colab disconnects.

**What this does:** trains **one epoch**. For each batch of 32 complaints it (1) turns the text into numbers, (2) makes predictions, (3) measures the
error, (4) works out how to change the model to reduce the error, and (5) takes a small step. It prints the average error every 200 steps.
If all epochs are done it just says so.

In [ ]:
if epoch_number >= EPOCHS:
    print("All epochs are finished. Go to the final cells.")
else:
    model.train()
    order = np.random.permutation(len(train))
    running_loss = 0.0
    start_time = time.time()

    for step in range(steps_per_epoch):
        rows = order[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
        texts = train["input_text"].iloc[rows].tolist()
        labels = torch.tensor(train["label"].iloc[rows].to_numpy(), dtype=torch.float32, device=device)

        batch = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_gpu):
            scores = model(**batch).logits.squeeze(-1)
            loss = loss_function(scores.float(), labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        if (step + 1) % 200 == 0:
            print(f"epoch {epoch_number + 1} step {step + 1}/{steps_per_epoch} | average loss {running_loss / 200:.4f} | {time.time() - start_time:.0f}s")
            running_loss = 0.0

    epoch_number += 1
    print(f"epoch {epoch_number} finished in {(time.time() - start_time) / 60:.1f} minutes")

**What this does:** predicts the **validation** complaints with the current model (no learning happens here), prints the two ranking scores,
and saves the probabilities to Drive as `val_predictions_distilbert_epoch1.csv` (or `roberta`, epoch 2, and so on).

In [ ]:
model.eval()
valid_probabilities = []

with torch.no_grad():
    for start in range(0, len(valid), 64):
        texts = valid["input_text"].iloc[start:start + 64].tolist()
        batch = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_gpu):
            scores = model(**batch).logits.squeeze(-1)
        valid_probabilities.extend(torch.sigmoid(scores.float()).cpu().numpy().tolist())

valid_probabilities = np.array(valid_probabilities)
pr_auc = average_precision_score(valid["label"], valid_probabilities)
roc_auc = roc_auc_score(valid["label"], valid_probabilities)
print(f"after epoch {epoch_number}: validation PR-AUC {pr_auc:.4f} | ROC-AUC {roc_auc:.4f}")
print("(for reference, the champion B4 had PR-AUC 0.4275 and ROC-AUC 0.7519 on the real validation set)")

result = pd.DataFrame({"complaint_id": valid["complaint_id"], "p_relief": valid_probabilities})
result.to_csv(OUT_DIR / f"val_predictions_{MODEL_TAG}_epoch{epoch_number}.csv", index=False)
print("saved", OUT_DIR / f"val_predictions_{MODEL_TAG}_epoch{epoch_number}.csv")

## Finish
Run the two cells above again until `epoch_number` equals `EPOCHS` (2). Then run the last cell.

**What this does:** saves the fine-tuned model (250 MB to 500 MB) to Drive so it can be reused without training again, and lists what is saved.
**Download the `val_predictions_..._epoch1.csv` and `..._epoch2.csv` files** from the `banksight_colab/outputs` Drive folder into your laptop's
`data/colab/outputs/` folder for notebook 08c.

In [ ]:
if epoch_number >= EPOCHS:
    model.save_pretrained(OUT_DIR / f"{MODEL_TAG}_relief")
    tokenizer.save_pretrained(OUT_DIR / f"{MODEL_TAG}_relief")
print("saved files:", sorted(p.name for p in OUT_DIR.iterdir()))

## How to explain this in an interview
> "I fine-tuned DistilBERT on a free Colab GPU with a plain PyTorch loop: mixed precision, a warm-up and decay learning-rate schedule, gradient clipping and
> fixed settings chosen before looking at validation results. I then brought its predictions back to compare against my champion model locally."